In [5]:
from nba_api.stats.endpoints import shotchartdetail
from nba_api.stats.static import teams
import time
import pandas as pd
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool, ColorBar, LinearColorMapper, Label
from bokeh.palettes import YlOrRd9, Viridis256
from bokeh.transform import linear_cmap
import numpy as np
from bokeh.plotting import figure, show
from bokeh.io import output_notebook

In [8]:
# Key events to annotate on charts
key_events = {
    1980: "3PT Line\nIntroduced",
    1995: "3PT Line\nShortened",
    1997: "3PT Line\nRestored",
    2010: "Curry\nDrafted",
    2014: "1st\nChampionship",
    2016: "73-9\nSeason",
    2016: "KD\nJoins",
    2019: "Last\nTitle Run",
    2020: "Klay & Steph\nInjured",
    2022: "4th\nChampionship",
}

# Era definitions (start_year, end_year, label)
eras = [
    (1979, 1994, "Early Adoption Era"),
    (1994, 1997, "Short Line Era"),
    (1997, 2015, "Mid Range Era"),
    (2015, 2024, "Warriors Dynasty (2015–2024)"),
    (2024, 2026, "Modern NBA"),
]

# Color scheme
GSW_COLOR    = "#FFC72C"   # Warriors = gold
LEAGUE_COLOR = "#45cf40"   # Gray = league
BG_COLOR     = "#0b1120"   # Dark bg
PANEL_COLOR  = "#111827"   # Slightly lighter panel
WHITE        = "#f0f4ff"
RED          = "#ef4444"
GREEN        = "#22c55e"
BLUE         = "#3b82f6"

print("Key events, eras and colors defined")
print(f"{len(key_events)} key events")
print(f"{len(eras)} eras")

Key events, eras and colors defined
9 key events
5 eras


In [6]:
# -----------------------------
# GET TEAM ID
# -----------------------------
nba_teams = teams.get_teams()
gsw = [t for t in nba_teams if t['abbreviation'] == 'GSW'][0]
GSW_TEAM_ID = gsw['id']

print(f"GSW Team ID: {GSW_TEAM_ID}")

# -----------------------------
# SEASONS
# -----------------------------
seasons = [
    '2014-15', '2015-16', '2016-17',
    '2017-18', '2018-19', '2022-23'
]

all_shots = []

# -----------------------------
# FETCH DATA
# -----------------------------
for season in seasons:
    print(f"\nPulling {season}...")

    try:
        shot_chart = shotchartdetail.ShotChartDetail(
            team_id=GSW_TEAM_ID,
            player_id=0,
            season_nullable=season,
            season_type_all_star='Regular Season',
            context_measure_simple='FGA'
        )

        df_shots = shot_chart.get_data_frames()[0]

        if df_shots.empty:
            print(f"⚠️ No data returned for {season}")
            continue

        df_shots['SEASON'] = season
        all_shots.append(df_shots)

        print(f"  ✅ {len(df_shots)} shots pulled")

        time.sleep(5)  # avoid rate limits

    except Exception as e:
        print(f"  ❌ Error for {season}: {e}")
        time.sleep(5)

# -----------------------------
# COMBINE ALL DATA
# -----------------------------
if not all_shots:
    raise ValueError("No data was fetched from NBA API.")

shots_df = pd.concat(all_shots, ignore_index=True)

print(f"\nTotal shots collected: {len(shots_df)}")

# -----------------------------
# FILTER 3PT SHOTS
# -----------------------------
shots_3pt = shots_df[
    shots_df['SHOT_TYPE'] == '3PT Field Goal'
].copy()

print(f"Total 3PT attempts: {len(shots_3pt)}")

# -----------------------------
# ZONE STATS
# -----------------------------
zone_stats = shots_3pt.groupby('SHOT_ZONE_AREA').agg(
    attempts=('SHOT_ATTEMPTED_FLAG', 'sum'),
    makes=('SHOT_MADE_FLAG', 'sum')
).reset_index()

zone_stats['efficiency'] = (zone_stats['makes'] / zone_stats['attempts']).round(3)
zone_stats['frequency'] = (zone_stats['attempts'] / zone_stats['attempts'].sum()).round(3)

print("\nZone breakdown:")
print(
    zone_stats.sort_values('frequency', ascending=False)
    .to_string(index=False)
)

GSW Team ID: 1610612744

Pulling 2014-15...
  ✅ 7137 shots pulled

Pulling 2015-16...
  ✅ 7159 shots pulled

Pulling 2016-17...
  ✅ 7140 shots pulled

Pulling 2017-18...
  ✅ 6979 shots pulled

Pulling 2018-19...
  ✅ 7361 shots pulled

Pulling 2022-23...
  ✅ 7393 shots pulled

Total shots collected: 43169
Total 3PT attempts: 16104

Zone breakdown:
       SHOT_ZONE_AREA  attempts  makes  efficiency  frequency
Right Side Center(RC)      4931   1968       0.399      0.306
 Left Side Center(LC)      4635   1788       0.386      0.288
            Center(C)      2992   1143       0.382      0.186
         Left Side(L)      1761    710       0.403      0.109
        Right Side(R)      1629    698       0.428      0.101
       Back Court(BC)       151      7       0.046      0.009


In [9]:
output_notebook()

zone_data = zone_stats[zone_stats['SHOT_ZONE_AREA'] != 'Back Court(BC)'].copy()

# Map zones to court coordinates (x, y, width, height)
# NBA court: x goes from -250 to 250, y from -50 to 420
zone_coords = {
    'Right Side(R)':        dict(x=220,   y=50,  w=60,  h=120),
    'Left Side(L)':         dict(x=-220,  y=50,  w=60,  h=120),
    'Right Side Center(RC)':dict(x=160,   y=200, w=100, h=160),
    'Left Side Center(LC)': dict(x=-160,  y=200, w=100, h=160),
    'Center(C)':            dict(x=0,     y=270, w=120, h=120),
}

# Add coords to zone_data
zone_data['x'] = zone_data['SHOT_ZONE_AREA'].map(lambda z: zone_coords.get(z.strip(), {}).get('x', 0))
zone_data['y'] = zone_data['SHOT_ZONE_AREA'].map(lambda z: zone_coords.get(z.strip(), {}).get('y', 0))
zone_data['w'] = zone_data['SHOT_ZONE_AREA'].map(lambda z: zone_coords.get(z.strip(), {}).get('w', 0))
zone_data['h'] = zone_data['SHOT_ZONE_AREA'].map(lambda z: zone_coords.get(z.strip(), {}).get('h', 0))
zone_data['freq_pct']    = (zone_data['frequency']  * 100).round(1)
zone_data['eff_pct']     = (zone_data['efficiency'] * 100).round(1)
zone_data['zone_label']  = zone_data['SHOT_ZONE_AREA'].str.replace(r'\(.*\)', '', regex=True).str.strip()

zone_source = ColumnDataSource(zone_data)

# Color mapper — frequency drives the color
mapper = LinearColorMapper(
    palette=YlOrRd9[::-1],
    low=zone_data['frequency'].min(),
    high=zone_data['frequency'].max()
)

# Court figure
court = figure(
    title="GSW 3-Point Shot Frequency & Efficiency by Zone (Dynasty Era 2014–2019, 2022-23)",
    width=600, height=580,
    x_range=(-260, 260),
    y_range=(-60, 430),
    tools="hover,save",
    toolbar_location="above"
)

# Styling
court.background_fill_color = PANEL_COLOR
court.border_fill_color     = BG_COLOR
court.outline_line_color    = "#1f2937"
court.title.text_color      = WHITE
court.title.text_font       = "Georgia"
court.title.text_font_size  = "12px"
court.xaxis.visible         = False
court.yaxis.visible         = False
court.xgrid.visible         = False
court.ygrid.visible         = False

LINE_COLOR = "#4b5563"
LW = 1.5

# Court boundary
court.rect(0, 190, 500, 420, fill_alpha=0,
           line_color=LINE_COLOR, line_width=LW)

# Paint / key
court.rect(0, 142.5, 160, 190, fill_color="#0d1b2a",
           fill_alpha=1, line_color=LINE_COLOR, line_width=LW)

# Free throw circle
theta = np.linspace(0, 2*np.pi, 100)
court.line(60*np.cos(theta), 190 + 60*np.sin(theta),
           color=LINE_COLOR, line_width=LW)

# Restricted area arc
ra_theta = np.linspace(0, np.pi, 50)
court.line(40*np.cos(ra_theta), 40*np.sin(ra_theta),
           color=LINE_COLOR, line_width=LW)

# Backboard
court.segment(-30, -7.5, 30, -7.5,
              line_color=LINE_COLOR, line_width=LW*2)

# Rim
rim_theta = np.linspace(0, 2*np.pi, 50)
court.line(10*np.cos(rim_theta), 10*np.sin(rim_theta),
           color="#ff6600", line_width=2)

# 3-point arc
arc_theta = np.linspace(0, np.pi, 200)
arc_x = 237.5 * np.cos(arc_theta)
arc_y = 237.5 * np.sin(arc_theta)
mask  = arc_y >= 0
court.line(arc_x[mask], arc_y[mask],
           color=LINE_COLOR, line_width=LW)

# Corner 3 lines
court.segment(-220, -52, -220, 92.5,
              line_color=LINE_COLOR, line_width=LW)
court.segment( 220, -52,  220, 92.5,
              line_color=LINE_COLOR, line_width=LW)

court.rect(
    x='x', y='y', width='w', height='h',
    source=zone_source,
    fill_color=linear_cmap('frequency', YlOrRd9[::-1],
                           zone_data['frequency'].min(),
                           zone_data['frequency'].max()),
    fill_alpha=0.75,
    line_color=WHITE, line_width=1
)

for _, row in zone_data.iterrows():
    # Zone name
    court.add_layout(Label(
        x=row['x'], y=row['y'] + 20,
        text=row['zone_label'],
        text_color=WHITE, text_font_size="9px",
        text_align="center", text_baseline="middle"
    ))
    # Frequency
    court.add_layout(Label(
        x=row['x'], y=row['y'],
        text=f"Freq: {row['freq_pct']}%",
        text_color=GSW_COLOR, text_font_size="9px",
        text_align="center", text_baseline="middle"
    ))
    # Efficiency
    court.add_layout(Label(
        x=row['x'], y=row['y'] - 20,
        text=f"Eff: {row['eff_pct']}%",
        text_color=GREEN, text_font_size="9px",
        text_align="center", text_baseline="middle"
    ))

color_bar = ColorBar(
    color_mapper=mapper,
    label_standoff=8,
    width=12,
    location=(0, 0),
    background_fill_color=PANEL_COLOR,
    major_label_text_color=LEAGUE_COLOR,
    title="Frequency",
    title_text_color=WHITE
)
court.add_layout(color_bar, 'right')

court.add_tools(HoverTool(tooltips=[
    ("Zone",       "@SHOT_ZONE_AREA"),
    ("Attempts",   "@attempts"),
    ("Makes",      "@makes"),
    ("Frequency",  "@freq_pct%"),
    ("Efficiency", "@eff_pct%"),
]))

show(court)

Loading BokehJS ...